In [0]:
from datetime import datetime
from pyspark.sql import Row

In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ------------------------------------
# Add Standard Metadata Columns
# ------------------------------------

def add_metadata(df, source_system):

    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file_name", col("_metadata.file_path"))
        .withColumn("load_id", expr("uuid()"))
        .withColumn("pipeline_run_id", expr("uuid()"))
        .withColumn("source_system", lit(source_system))
    )

In [0]:
def get_duplicate_count(df, key):

    return (
        df.groupBy(key)
          .count()
          .filter(col("count") > 1)
          .count()
    )


def get_null_count(df, column_name):

    return (
        df.filter(col(column_name).isNull()).count()
    )

In [0]:
from pyspark.sql import Row

from datetime import datetime
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)

def write_audit(
        pipeline_name,
        table_name,
        load_type,
        rows_read,
        rows_written,
        status,
        error_message,
        pipeline_run_id):

    current_time = datetime.now()

    audit_schema_def = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_read", IntegerType(), True),
        StructField("rows_written", IntegerType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("pipeline_run_id", StringType(), True)
    ])

    audit_data = [(
        str(pipeline_name),
        str(table_name),
        str(load_type),
        current_time,
        current_time,
        int(rows_read),
        int(rows_written),
        str(status),
        str(error_message),
        str(pipeline_run_id)
    )]

    audit_df = spark.createDataFrame(
        audit_data,
        schema=audit_schema_def
    )

    audit_table = f"{catalog_name}.{audit_schema}.pipeline_audit"

    (
        audit_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(audit_table)
    )

# def write_audit(
#         pipeline_name,
#         table_name,
#         load_type,
#         rows_read,
#         rows_written,
#         status,
#         error_message,
#         pipeline_run_id):

#     current_time = datetime.now()

#     audit_df = spark.createDataFrame([

#         Row(
#             pipeline_name=pipeline_name,
#             table_name=table_name,
#             load_type=load_type,
#             start_time=current_time,
#             end_time=current_time,
#             rows_read=int(rows_read),
#             rows_written=int(rows_written),
#             status=status,
#             error_message=error_message,
#             pipeline_run_id=pipeline_run_id
#         )

#     ])

#     audit_table = f"{catalog_name}.{audit_schema}.pipeline_audit"

#     (
#         audit_df.write
#             .mode("append")
#             .saveAsTable(audit_table)
#     )

In [0]:
%sql
-- CREATE DATABASE IF NOT EXISTS insurance_audit;

In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS insurance_audit.pipeline_audit
-- (
-- pipeline_name STRING,
-- table_name STRING,
-- load_type STRING,
-- start_time TIMESTAMP,
-- end_time TIMESTAMP,
-- rows_read BIGINT,
-- rows_written BIGINT,
-- status STRING,
-- error_message STRING,
-- pipeline_run_id STRING
-- )
-- USING DELTA;